# Kaggle smoke — dùng CLI của repository
Bật Internet + GPU. Chạy theo thứ tự. Training/inference và MLflow verification/export đều nằm trong repo.
Utility phải được commit/push lên GitHub main trước khi chạy notebook. Số liệu bảng lấy trực tiếp từ MLflow API.


In [ ]:
from pathlib import Path
import subprocess, os, sys
repo = Path('/kaggle/working/math-instruct-llama')
url = 'https://github.com/nhannhan2201/math-instruct-llama.git'
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', url, str(repo)], check=True)
else:
    assert subprocess.check_output(['git','remote','get-url','origin'], cwd=repo, text=True).strip() == url
    assert not subprocess.check_output(['git','status','--porcelain'], cwd=repo, text=True).strip(), 'Working tree dirty'
    subprocess.run(['git','switch','main'], cwd=repo, check=True)
    subprocess.run(['git','pull','--ff-only','origin','main'], cwd=repo, check=True)
os.chdir(repo)
subprocess.run(['git','log','-1','--oneline'], check=True)
assert Path('src/smoke_review.py').exists(), 'Push utility to GitHub main first'


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
# Workaround torchao cũ đã dùng trên Kaggle; không thay torch/CUDA.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], check=True)
check = subprocess.run([sys.executable,'-m','pip','check'], capture_output=True, text=True)
Path('/kaggle/working/smoke-pip-check.log').write_text(check.stdout + check.stderr)
print('pip check:', check.returncode, check.stdout, check.stderr)


In [ ]:
subprocess.run([sys.executable, '-m', 'src.smoke_review'], check=True)


Tải `/kaggle/working/smoke-review-*.zip` trong Kaggle Output trước reset, cùng `smoke-pip-check.log`.
ZIP có SQLite backup, MLflow store, outputs, manifest, logs và verification.json/checksums.
Chỉ VERIFIED khi CLI báo PASS; reload demo không phải answer accuracy hay numerical tolerance.
